In [0]:
%run ../00_common/data_utils

In [0]:
def update_ukey_to_master_table(update_df):
    # 更新 master recode 中ukey 
    base_table = DeltaTable.forName(spark, f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")

    # merge
    # regular consumer 和 rebind consumer 进行不同主键的merger
    (base_table.alias("b")
        .merge(update_df.alias("u"), 
            f"""
                (u.match_type = '{MATCH_TYPE_REGULAR_STR}' and b.scon_mrkt_code = u.mrkt_code and b.consumermdmkey = u.master_consumermdmkey) or
                (u.match_type = '{MATCH_TYPE_REBIND_STR}'  and b.scon_mrkt_code = u.mrkt_code and b.scon_srcc_id = u.srcc_id)
            """
        )
        .whenMatchedUpdate(
            set = {
                "consumermdmkey": F.col("u.new_consumermdmkey"),
                "task_id": F.col("u.task_id"),
            }
        )
        .execute())

# def delete_ukey_to_derived_table(delete_df):
#     base_table_l1 = DeltaTable.forName(spark, f"{get_env_config('golden_consumer_master_database')}.t_derived_consumer_l1")
#     base_table_l2 = DeltaTable.forName(spark, f"{get_env_config('golden_consumer_master_database')}.t_derived_consumer_l2")
#     base_table_l3 = DeltaTable.forName(spark, f"{get_env_config('golden_consumer_master_database')}.t_derived_consumer_l3")

#     base_table_list = [base_table_l1, base_table_l2, base_table_l3]

#     for base_table in base_table_list:
#         (base_table.alias("b")
#             .merge(delete_df.alias("u"), 
#                 (F.col("b.scon_mrkt_code") == F.col("u.mrkt_code")) &
#                 (F.col("b.consumermdmkey") == F.col("u.master_consumermdmkey"))
#             )
#             .whenMatchedDelete()
#             .execute()
#         )


# def delete_ukey_to_cbr_table(delete_df):
#     base_table_l1 = DeltaTable.forName(spark, f"{get_env_config('golden_consumer_combine_database')}.t_cbr_dataset")
#     base_table_l2 = DeltaTable.forName(spark, f"{get_env_config('golden_consumer_combine_database')}.t_cbr_withoutPII_dataset")

#     base_table_list = [base_table_l1, base_table_l2]

#     for base_table in base_table_list:
#         (base_table.alias("b")
#             .merge(delete_df.alias("u"), 
#                 (F.col("b.MarketCode") == F.col("u.mrkt_code")) &
#                 (F.col("b.MDMKey") == F.col("u.master_consumermdmkey"))
#             )
#             .whenMatchedDelete()
#             .execute()
#         )

In [0]:
def ukey_process(task_id):
    new_ukey_df = (get_ukey_group_by_process(task_id)
        .filter(F.col("is_master_recode") == True)
        .filter(F.coalesce(F.col("master_consumermdmkey"), F.lit("")) != F.coalesce(F.col("new_consumermdmkey"), F.lit("")))
        .select(
            F.col("task_id"), 
            F.col("mrkt_code"), 
            F.col("master_consumermdmkey"), 
            F.col("new_consumermdmkey"), 
            F.when(F.col("match_type") == MATCH_TYPE_REGULAR_STR, F.lit(None)).otherwise(F.col("srcc_id")).alias("srcc_id"),
            F.col("match_type")   
        )
        .distinct())
    # display(new_ukey_df)
    
    # 1. update t_master_consumer ukey 
    update_ukey_to_master_table(new_ukey_df)
    
    # 2. delete t_derived_consumer_l1、t_derived_consumer_l2、t_derived_consumer_l3 ukey
    # delete_ukey_to_derived_table(new_ukey_df.filter(F.col("match_type") == MATCH_TYPE_REGULAR_STR))

    # 3. delete t_cbr_dataset、t_cbr_withoutPII_dataset  ukey
    #  同步删除到c表
    # delete_ukey_to_cbr_table(new_ukey_df.filter(F.col("match_type") == MATCH_TYPE_REGULAR_STR))

In [0]:
task_id = dbutils.widgets.get("task_id")
print(f"task_id: {task_id}")

with StepLogger("4.4_ukey_update&delete", "04-4", "consumerlist", task_id=task_id) as logger:
    ukey_process(task_id)

In [0]:
# Pass to append c table: t_cbr_dataset、t_cbr_withoutPII_dataset
# t_cbr_dataset_name = f"{get_env_config('golden_consumer_combine_database')}.t_cbr_dataset"
# t_cbr_dataset_version = get_latest_version(t_cbr_dataset_name)
# dbutils.jobs.taskValues.set(key = 'c_cbr_dataset_versions', value = f"{t_cbr_dataset_name}:{t_cbr_dataset_version}")

# t_cbr_withoutPII_dataset_name = f"{get_env_config('golden_consumer_combine_database')}.t_cbr_withoutPII_dataset"
# t_cbr_withoutPII_dataset_version = get_latest_version(t_cbr_withoutPII_dataset_name)
# dbutils.jobs.taskValues.set(key = 'c_cbr_withoutPII_dataset_versions', value = f"{t_cbr_withoutPII_dataset_name}:{t_cbr_withoutPII_dataset_version}")

# output_params = {
#     "c_cbr_dataset_versions": f"{t_cbr_dataset_name}:{t_cbr_dataset_version}",
#     "c_cbr_withoutPII_dataset_versions": f"{t_cbr_withoutPII_dataset_name}:{t_cbr_withoutPII_dataset_version}"
# }
# dbutils.notebook.exit(json.dumps(output_params))